# Getting Started with pydantic-ai-cloudflare

Go from zero to a working AI agent in 5 minutes.

**What you need:** A free Cloudflare account ([sign up](https://dash.cloudflare.com/sign-up))

In [ ]:
!pip install pydantic-ai-cloudflare

## Set up credentials

1. Get your **Account ID** from [dash.cloudflare.com](https://dash.cloudflare.com) (right sidebar)
2. Create an **API Token** at [API Tokens](https://dash.cloudflare.com/profile/api-tokens) with Workers AI Read + Browser Rendering Edit

In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

## Simple agent

In [2]:
from pydantic_ai_cloudflare import cloudflare_agent

agent = cloudflare_agent()
result = agent.run_sync("What is Cloudflare Workers AI in one sentence?")
print(result.output)

Cloudflare Workers AI is a platform that allows developers to build and deploy serverless applications with integrated artificial intelligence and machine learning capabilities at the edge of the network, closest to users.


## Structured output

In [3]:
from pydantic import BaseModel

class CompanyProfile(BaseModel):
    name: str
    industry: str
    founded_year: int
    products: list[str]

agent = cloudflare_agent(output_type=CompanyProfile)
result = agent.run_sync("Tell me about Cloudflare")
c = result.output

print(f"Name: {c.name}")
print(f"Industry: {c.industry}")
print(f"Founded: {c.founded_year}")
print(f"Products: {c.products}")
print(f"Type: {type(c).__name__}")

Name: Cloudflare
Industry: Technology
Founded: 2009
Products: ['Content Delivery Network', 'DDoS Protection', 'SSL/TLS Encryption', 'Domain Name System']
Type: CompanyProfile


## Complex structured output

For 4+ nested models with Literal types, use `cf_structured()` which handles Workers AI quirks.

In [4]:
from typing import Literal
from pydantic_ai_cloudflare import cf_structured_sync

class NewsItem(BaseModel):
    headline: str
    summary: str

class TriggerEvent(BaseModel):
    event: str
    priority: Literal["HIGH", "MEDIUM", "LOW"]

class HiringSignal(BaseModel):
    role: str
    insight: str

class Company(BaseModel):
    name: str
    industry: str
    employees: str

class MarketIntel(BaseModel):
    news: list[NewsItem]
    triggers: list[TriggerEvent]
    hiring: list[HiringSignal]

class Positioning(BaseModel):
    value_prop: str
    strengths: list[str]

class TechAssessment(BaseModel):
    score: int
    signals: list[str]

class NextStep(BaseModel):
    action: str
    priority: Literal["HIGH", "MEDIUM", "LOW"]

class Report(BaseModel):
    overview: str
    company: Company
    market: MarketIntel
    positioning: Positioning
    tech: TechAssessment
    next_steps: list[NextStep]

result = cf_structured_sync(
    "Report on a fictional company 'CloudSync' that provides cloud storage "
    "for enterprises. 2000 employees. 3 news, 3 triggers, 2 hiring, 4 steps.",
    Report,
)

print(f"Company: {result.company.name} ({result.company.industry}), {result.company.employees} employees")
print(f"\nNews: {len(result.market.news)} items")
for n in result.market.news:
    print(f"  - {n.headline}")
print(f"\nTriggers: {len(result.market.triggers)} events")
for t in result.market.triggers:
    print(f"  [{t.priority}] {t.event}")
print(f"\nNext steps: {len(result.next_steps)} actions")
for s in result.next_steps:
    print(f"  [{s.priority}] {s.action}")

Company: CloudSync (Cloud Storage), 2000 employees

News: 3 items
  - CloudSync Expands Operations to Europe
  - CloudSync Partners with Microsoft to Enhance Security
  - CloudSync Raises $100M in Funding Round

Triggers: 3 events
  [HIGH] Expansion into new markets
  [MEDIUM] Partnership with major tech company
  [LOW] Significant funding round

Next steps: 4 actions
  [HIGH] Schedule a meeting with the sales team
  [MEDIUM] Conduct a technical assessment
  [LOW] Research competitors
  [MEDIUM] Develop a customized pitch


## Model discovery

In [5]:
from pydantic_ai_cloudflare import list_models, recommend_model

models = list_models()
print(f"{len(models)} models available:")
for m in models[:5]:
    print(f"  {m['name']}: {m['context']} context, {m['speed']}")

print(f"\nRecommended for reasoning: {recommend_model(task='reasoning')}")
print(f"Recommended for vision: {recommend_model(task='vision')}")

9 models available:
  Llama 3.3 70B: 128K context, fast
  Llama 3.1 8B: 128K context, very_fast
  Qwen 3 30B: 128K context, fast
  Kimi K2.6: 256K context, medium
  Gemma 4 26B: 128K context, fast

Recommended for reasoning: @cf/qwen/qwen3-30b-a3b-fp8
Recommended for vision: @cf/google/gemma-4-26b-a4b-it


## What's next

- [02_web_research.ipynb](02_web_research.ipynb) — Browse websites and extract structured data
- [03_rag_pipeline.ipynb](03_rag_pipeline.ipynb) — Full RAG with Vectorize
- [04_persistent_chat.ipynb](04_persistent_chat.ipynb) — Multi-session conversations